# Очистка данных

**План**
1. Загрузка данных
2. Тип данных event_time
3. Флаг дублей
4. Нулевые и отрицательные цены
5. Проверка результата
6. Сохранение
7. Итоги

Решения берём из раздела 7 ноутбука 01_EDA. Порядок шагов важен: флаг дублей ставим до замены цен. Если сделать наоборот, две разные строки с ценами -5 и -3 станут идентичными (обе NaN) и получат флаг дубля, хотя это разные события.

## 1. Загрузка

In [11]:
import glob
import os
import numpy as np
import pandas as pd

In [12]:
files = sorted(glob.glob('../data/raw/*.csv'))

dfs = []
for file_name in files:
    small_df = pd.read_csv(file_name)
    small_df['source_file'] = os.path.basename(file_name)
    print(f'{file_name}: {len(small_df):,} строк')
    dfs.append(small_df)

df = pd.concat(dfs, ignore_index=True)
del dfs
df

../data/raw\2019-Dec.csv: 3,533,286 строк
../data/raw\2019-Nov.csv: 4,635,837 строк
../data/raw\2019-Oct.csv: 4,102,283 строк
../data/raw\2020-Feb.csv: 4,156,682 строк
../data/raw\2020-Jan.csv: 4,264,752 строк


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,source_file
0,2019-12-01 00:00:00 UTC,remove_from_cart,5712790,1487580005268456287,NaN,f.o.x,6.27,576802932,51d85cb0-897f-48d2-918b-ad63965c12dc,2019-Dec.csv
1,2019-12-01 00:00:00 UTC,view,5764655,1487580005411062629,NaN,cnd,29.05,412120092,8adff31e-2051-4894-9758-224bfa8aec18,2019-Dec.csv
2,2019-12-01 00:00:02 UTC,cart,4958,1487580009471148064,NaN,runail,1.19,494077766,c99a50e8-2fac-4c4d-89ec-41c05f114554,2019-Dec.csv
3,2019-12-01 00:00:05 UTC,view,5848413,1487580007675986893,NaN,freedecor,0.79,348405118,722ffea5-73c0-4924-8e8f-371ff8031af4,2019-Dec.csv
4,2019-12-01 00:00:07 UTC,view,5824148,1487580005511725929,NaN,NaN,5.56,576005683,28172809-7e4a-45ce-bab0-5efa90117cd5,2019-Dec.csv
...,...,...,...,...,...,...,...,...,...,...
20692835,2020-01-31 23:59:52 UTC,remove_from_cart,5886774,1487580006317032337,NaN,NaN,1.59,607092857,a4ccd1c4-a9d2-48d0-9816-082ec5bb5e47,2020-Jan.csv
20692836,2020-01-31 23:59:52 UTC,remove_from_cart,5886774,1487580006317032337,NaN,NaN,1.59,607092857,a4ccd1c4-a9d2-48d0-9816-082ec5bb5e47,2020-Jan.csv
20692837,2020-01-31 23:59:53 UTC,view,5875432,2084144451428549153,NaN,NaN,2.05,423651741,fb42a963-abef-4c4f-b1ba-f5812dd54e80,2020-Jan.csv
20692838,2020-01-31 23:59:57 UTC,remove_from_cart,5820745,1487580006317032337,NaN,NaN,2.22,607092857,a4ccd1c4-a9d2-48d0-9816-082ec5bb5e47,2020-Jan.csv


## 2. Тип данных event_time

**Что делаем**
- Переводим поле "event_time" из строки во временной тип

In [13]:
df['event_time'] = pd.to_datetime(df['event_time'], format='%Y-%m-%d %H:%M:%S UTC', utc=True)

dtype = df['event_time'].dtype
start = df['event_time'].min()
end = df['event_time'].max()
print(f'Тип: {dtype}')
print(f'Период: {start} - {end}')

Тип: datetime64[us, UTC]
Период: 2019-10-01 00:00:00+00:00 - 2020-02-29 23:59:59+00:00


**Промежуточный вывод**
- Поле "event_time" перевели во временной тип с часовым поясом UTC

## 3. Флаг дублей

**Что делаем**
- Добавляем поле "is_duplicate", как флаг: первая строка из группы одинаковых получает False, повторные копии True

In [ ]:
cols = df.columns.drop('source_file')

df['is_duplicate'] = df.duplicated(subset=cols, keep='first')

df.head(15)

In [ ]:
duplicates_left = df.loc[~df['is_duplicate']].duplicated(subset=cols).sum()
print(f'Дублей среди строк с is_duplicate = False: {duplicates_left}')

Дублей среди строк с is_duplicate = False: 0


**Промежуточный вывод**
- Дубли пометили флагом, строки сохранили

## 4. Нулевые и отрицательные цены

**Что делаем**
- Заменяем отрицательные и нулевые цены на NaN.

In [ ]:
negative = df['price'] < 0
zero = df['price'] == 0

total_sum = negative.sum() + zero.sum()

print(f'Всего отрицательных и нелогичных цен: {total_sum:,}')

Всего отрицательных и нелогичных цен: 104,288


In [ ]:
nonpos = negative | zero

df.loc[nonpos, 'price'] = np.nan

n_price_null = int(df['price'].isna().sum())
print(f'Пропусков в price после замены: {n_price_null:,}')

Пропусков в price после замены: 104,288


**Промежуточный вывод**
- Нулевые и отрицательные цены заменили на NaN, строки сохранили

## 5. Проверка результата

**Что проверяем**
- Флаг дублей, тип времени, цены
- Пропуски в остальных полях не изменились 

In [ ]:
assert df['is_duplicate'].dtype == bool
assert df['event_time'].isna().sum() == 0
assert df['price'].min() > 0
assert df['price'].isna().sum() == total_sum
print('Все проверки пройдены')

Все проверки пройдены


In [ ]:
missing = (
    df.isna().agg(['sum', 'mean']).T
    .rename(columns={'sum': 'n_missing', 'mean': 'share_pct'})
)
missing['n_missing'] = missing['n_missing'].astype(int)
missing['share_pct'] *= 100

missing

,n_missing,share_pct
event_time,0,0.000000
event_type,0,0.000000
product_id,0,0.000000
category_id,0,0.000000
category_code,20339246,98.291225
brand,8757117,42.319551
price,104288,0.503981
user_id,0,0.000000
user_session,4598,0.022220
source_file,0,0.000000


**Промежуточный вывод**
- Все изменения прошли проверку

## 6. Сохранение

Сохраняем в parquet: формат хранит типы данных (время остаётся временем, флаг остаётся булевым), читается намного быстрее CSV и занимает меньше места.

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

out_path = '../data/processed/events_clean.parquet'
df.to_parquet(out_path, index=False)
print(f'Сохранено: {out_path}, строк: {len(df):,}')

Сохранено: ../data/processed/events_clean.parquet, строк: 20,692,840


## 7. Итоги очистки

1. Поле "event_time" переведено во временной тип (UTC)
2. Пропуски оставлены без изменений
3. Дубли помечены флагом "is_duplicate", строки не удалены
4. Нулевые и отрицательные цены заменены на NaN